# RAG: Baseline Retrieval vs. Cross-Encoder Re-Ranking
**Course:** Language Models and Agents — RAG Assignment

**Document:** EU Artificial Intelligence Act — Regulation (EU) 2024/1689 (Official Journal version, 12.7.2024, 144 pages)

This notebook builds two retrieval pipelines over the **same chunked corpus** and compares them:

| | Pipeline 1 — Baseline | Pipeline 2 — Re-Ranking |
|---|---|---|
| Stage 1 | Bi-encoder (`all-MiniLM-L6-v2`) → FAISS → **Top 10** | *same Top 10* |
| Stage 2 | take Top 3 as-is | Cross-encoder (`ms-marco-MiniLM-L-6-v2`) re-scores 10 → **Top 3** |
| Generation | Gemini, identical prompt → **Answer 1** | Gemini, identical prompt → **Answer 2** |

**Why the AI Act?** It is long, heavily cross-referenced, and full of *prohibitions* (Art. 5), *exceptions* ("unless", "except where"),
and *recitals* that repeat the vocabulary of the binding articles — natural distractors for a bi-encoder.

**Notebook outline**
1. Setup & configuration
2. Load and clean the PDF
3. Structure-aware chunking
4. Pipeline 1 — bi-encoder index + Top-10 retrieval
5. Pipeline 2 — cross-encoder re-ranking
6. LLM generation (identical prompt for both pipelines)
7. Side-by-side comparison helper + smoke test
8. Failure-mode stress tests (rank-shift analysis + Answer 1 vs Answer 2) and 8b. aggregate evaluation over 73 labelled queries
9. Latency profiling
10. Scaling experiment: why not cross-encode the whole corpus?

## 1. Setup & configuration

In [1]:
# Install once (uncomment if running in a fresh environment, e.g. Colab):
# !pip install sentence-transformers faiss-cpu pymupdf google-genai python-dotenv pandas

import os, re, time, json, random
import numpy as np
import pandas as pd
import pymupdf                      # PDF text extraction (PyMuPDF)
import faiss                        # vector index
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from dotenv import load_dotenv
from google import genai
from google.genai import types

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

# Reproducibility: fix random seeds (both models run in inference mode, so results are deterministic anyway).
random.seed(0); np.random.seed(0); torch.manual_seed(0)

# ---- Configuration -------------------------------------------------------
PDF_PATH        = "data/EU_AI_Act_2024_1689.pdf"
BI_ENCODER      = "sentence-transformers/all-MiniLM-L6-v2"   # dense bi-encoder (384-d)
CROSS_ENCODER   = "cross-encoder/ms-marco-MiniLM-L-6-v2"     # re-ranker
TOP_K_RETRIEVE  = 10      # candidates from the vector store
TOP_K_CONTEXT   = 3       # chunks passed to the LLM
MAX_TOKENS      = 220     # max word-pieces per chunk (the bi-encoder truncates input at 256)
OVERLAP_TOKENS  = 40      # overlap between consecutive windows of the same article/recital

load_dotenv(".env")                                      # reads GEMINI_API_KEY from .env (never committed)
LLM_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.7-flash")
LLM_PAUSE_S = 6                                          # pause between LLM calls (free tier: per-minute and 20/day limits)
print("torch", torch.__version__, "| device: cpu | LLM:", LLM_MODEL)

C:\my_workspace\LMA\Assignment-1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch 2.14.1+cpu | device: cpu | LLM: gemini-3.7-flash


## 2. Load and clean the PDF
Two kinds of noise are removed before chunking:
* **Running headers/footers** on every page (`OJ L, 12.7.2024`, `EN`, the ELI link and `n/144`).
* **Page footnotes** (e.g. `(15) Regulation (EU) 2022/2065 …`). Their markers look exactly like recital numbers, which would
  corrupt the recital boundaries. They are detected by **font size**: body text is 9.6pt, footnotes are 8.5pt.
* **Superscripts**: footnote references inside sentences (`Committee (¹)`) are dropped, while exponents are kept as `10^25`
  — plain text extraction would otherwise turn the systemic-risk threshold of 10²⁵ FLOPs (Article 51) into "1025".

In [2]:
doc = pymupdf.open(PDF_PATH)
print("Pages:", len(doc))

HEADER_PATTERNS = [
    r"^OJ L, \d{1,2}\.\d{1,2}\.\d{4}$",
    r"^EN$",
    r"^ELI: http://data\.europa\.eu/eli/reg/2024/1689/oj$",
    r"^\d+/144$",
]
header_re = re.compile("|".join(HEADER_PATTERNS))

BODY_FONT_MIN = 9.0   # body text is 9.6pt; footnotes are 8.5pt; superscript references ~5pt

def clean_page(page) -> str:
    # Read the page line by line together with its font size, so page footnotes can be dropped.
    # Recital numbers "(12)" are also printed at 8.5pt, but they are followed by body-size text, whereas a
    # footnote marker is followed by more small text. Everything small after the first footnote marker is footnote.
    lines = []
    for block in page.get_text("dict")["blocks"]:
        for line in block.get("lines", []):
            spans = line["spans"]
            size = max([s["size"] for s in spans if not s["flags"] & 1] or [0])   # ignore superscripts
            text = ""
            for s in spans:
                if s["flags"] & 1 and size >= BODY_FONT_MIN:    # superscript inside a body-text line
                    if re.search(r"\b10$", text):               # exponent, e.g. 10^25 FLOPs (Article 51)
                        text += "^" + s["text"]
                    continue                                    # otherwise a footnote reference -> drop
                text += s["text"]
            if size >= BODY_FONT_MIN:
                text = re.sub(r" ?\(\s*\)", "", text)           # "(¹)" leaves an empty "()" behind
            if text.strip():
                lines.append((text.rstrip(), size))
    kept, in_footnotes = [], False
    for i, (text, size) in enumerate(lines):
        if header_re.match(text.strip()) or size < 6:       # running header/footer, superscript reference
            continue
        small = size < BODY_FONT_MIN
        next_small = i + 1 < len(lines) and lines[i + 1][1] < BODY_FONT_MIN
        if small and re.fullmatch(r"\(\d{1,3}\)", text.strip()) and next_small:
            in_footnotes = True
        if small and in_footnotes:
            continue
        kept.append(text)
    return "\n".join(kept)

raw_text = "\n".join(clean_page(p) for p in doc)
print(f"{len(raw_text.split()):,} words after cleaning")
print(raw_text[:600])

Pages: 144


88,129 words after cleaning
REGULATION (EU) 2024/1689 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL
of 13 June 2024
laying down harmonised rules on artificial intelligence and amending Regulations (EC) No 300/2008,
(EU) No 167/2013, (EU) No 168/2013, (EU) 2018/858, (EU) 2018/1139 and (EU) 2019/2144 and
Directives 2014/90/EU, (EU) 2016/797 and (EU) 2020/1828 (Artificial Intelligence Act)
(Text with EEA relevance)
THE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROPEAN UNION,
Having regard to the Treaty on the Functioning of the European Union, and in particular Articles 16 and 114 thereof,
Having regard to the proposal fr


## 3. Structure-aware chunking
The Act has three parts, each with a clear marker in the text:

| Part | Marker | Unit label |
|---|---|---|
| Preamble (180 recitals — *why* the rules exist) | `(1)`, `(2)` … on their own line | `Recital N` |
| Enacting terms (113 articles — the binding rules) | `Article N` + title line | `Article N — Title` |
| Annexes I–XIII (lists, e.g. high-risk use cases) | `ANNEX I` + title line | `Annex I — Title` |

**Chunking strategy**
1. Split the text into these logical units first, so a chunk never mixes two articles.
2. Units longer than `MAX_TOKENS` word-pieces are split into sentence-aligned windows with ~`OVERLAP_TOKENS` of overlap.
3. Each chunk is prefixed with its unit label (e.g. `[Article 5 — Prohibited AI practices]`) so both models see where the text comes from.

`MAX_TOKENS = 220` is chosen because `all-MiniLM-L6-v2` **silently truncates input after 256 word-pieces**. Measuring chunk
size with the model's own tokenizer (instead of counting words) guarantees that no clause at the end of a chunk is cut off.
The cross-encoder accepts 512 word-pieces, so query + chunk always fits.
Both pipelines use exactly this chunk list.

In [3]:
split_at = raw_text.index("HAVE ADOPTED THIS REGULATION")
annex_at = re.search(r"\nANNEX I\n", raw_text).start()
preamble, articles_txt, annexes_txt = raw_text[:split_at], raw_text[split_at:annex_at], raw_text[annex_at:]

def norm(s: str) -> str:
    # Join PDF line breaks into flowing text; fix hyphenation across lines.
    s = re.sub(r"-\n(?=[a-z])", "", s)
    return re.sub(r"\s+", " ", s).strip()

units = []   # (label, text)

# --- Recitals: "(N)" alone on a line starts recital N ----------------------
# Footnote markers in the preamble look identical ("(1)" on its own line), so a marker is only
# accepted if it is the *next* recital number in sequence (1, 2, 3, ... 180).
whereas = preamble[preamble.index("Whereas:"):]
starts, expected = [], 1
for m in re.finditer(r"\n\((\d{1,3})\)\n", whereas):
    if int(m.group(1)) == expected:
        starts.append(m); expected += 1
for m, nxt in zip(starts, starts[1:] + [None]):
    body = whereas[m.end(): nxt.start() if nxt else len(whereas)]
    body = re.split(r"\n\(\d{1,2}\)\n(?=OJ |Regulation |Directive |Council |Decision |Commission )", body)[0]  # drop page footnotes
    units.append((f"Recital {m.group(1)}", norm(body)))

def full_title(title, body):
    # Long titles wrap onto a second line: continue while the title ends in a function word
    # ("... an assessment of the") or the next line starts in lower case ("... in the AI regulatory" / "sandbox").
    while True:
        nxt, _, rest = body.partition("\n")
        if nxt and (re.search(r"\b(the|of|for|in|with|on|to|and|or|a|an|their|its)$", title.strip()) or nxt[:1].islower()):
            title, body = f"{title.strip()} {nxt.strip()}", rest
        else:
            return title.strip().rstrip("`"), body

# --- Articles: "Article N" line followed by its title line ------------------
parts = re.split(r"\nArticle (\d{1,3})\n([^\n]+)\n", articles_txt)
for num, title, body in zip(parts[1::3], parts[2::3], parts[3::3]):
    title, body = full_title(title, body)
    body = re.split(r"\n(?:CHAPTER [IVX]+|SECTION \d+)\n", body)[0]   # drop trailing chapter/section headings
    units.append((f"Article {num} — {title}", norm(body)))

# --- Annexes: "ANNEX X" line followed by its title line ---------------------
parts = re.split(r"\nANNEX ([IVX]+)\n([^\n]+)\n", annexes_txt)
for num, title, body in zip(parts[1::3], parts[2::3], parts[3::3]):
    title, body = full_title(title, body)
    units.append((f"Annex {num} — {title}", norm(body)))

n_rec = sum(u[0].startswith("Recital") for u in units)
n_art = sum(u[0].startswith("Article") for u in units)
n_anx = sum(u[0].startswith("Annex") for u in units)
print(f"Units: {n_rec} recitals, {n_art} articles, {n_anx} annexes")

Units: 180 recitals, 113 articles, 13 annexes


In [4]:
# Chunk size is measured in the bi-encoder's own word-pieces (not words), so no chunk can exceed its window.
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(BI_ENCODER)
n_tok = lambda s: len(tokenizer.tokenize(s))

def split_unit(label, text, max_tokens=MAX_TOKENS, overlap=OVERLAP_TOKENS):
    # Greedy sentence packing: sentences (split after . ; :) are added until the window is full;
    # the next window re-starts with the trailing sentences (~`overlap` tokens) of the previous one.
    budget = max_tokens - n_tok(f"[{label}] ")
    sents = []
    for s in re.split(r"(?<=[.;:])\s+", text):
        words = s.split()
        while n_tok(" ".join(words)) > budget:          # hard-split an over-long sentence
            cut = len(words) // 2
            sents.append(" ".join(words[:cut])); words = words[cut:]
        sents.append(" ".join(words))
    windows, cur, cur_tok = [], [], 0
    for s in sents:
        t = n_tok(s)
        if cur and cur_tok + t > budget:
            windows.append(" ".join(cur))
            tail, tail_tok = [], 0                      # carry overlap sentences forward
            for prev in reversed(cur):
                pt = n_tok(prev)
                if tail_tok + pt > overlap: break
                tail.insert(0, prev); tail_tok += pt
            cur, cur_tok = tail, tail_tok
        cur.append(s); cur_tok += t
    if cur:
        windows.append(" ".join(cur))
    return windows

chunks = []   # list of dicts: id, label, part, text
for label, text in units:
    for part_no, window in enumerate(split_unit(label, text)):
        chunks.append({"id": len(chunks), "label": label, "part": part_no,
                       "text": f"[{label}] {window}"})

chunk_texts = [c["text"] for c in chunks]
lens = np.array([len(t.split()) for t in chunk_texts])
print(f"{len(chunks)} chunks | words per chunk: mean {lens.mean():.0f}, max {lens.max()}")
pd.DataFrame(chunks).sample(5, random_state=1)[["id", "label", "text"]]

715 chunks | words per chunk: mean 134, max 195


,id,label,text
185,185,Recital 111,[Recital 111] It is appropriate to establish a methodology for the classification of general-purpose AI models as ge...
333,333,Article 6 — Classification rules for high-risk AI systems,[Article 6 — Classification rules for high-risk AI systems] 8. Any amendment to the conditions laid down in paragrap...
120,120,Recital 65,[Recital 65] This is to ensure that the deployer is aware and takes them into account when using the high-risk AI sy...
8,8,Recital 7,"[Recital 7] In order to ensure a consistent and high level of protection of public interests as regards health, safe..."
692,692,Annex VIII — Information to be submitted upon the registration of high-risk AI systems in accordance with Article 49,[Annex VIII — Information to be submitted upon the registration of high-risk AI systems in accordance with Article 4...


In [5]:
# Sanity check: no chunk should exceed the bi-encoder's 256 word-piece window.
n_tokens = np.array([len(tokenizer(t)["input_ids"]) for t in chunk_texts])   # includes [CLS]/[SEP]
print(f"word-pieces per chunk: mean {n_tokens.mean():.0f}, max {n_tokens.max()} | "
      f"chunks over 256: {(n_tokens > 256).sum()} of {len(chunks)}")

word-pieces per chunk: mean 168, max 222 | chunks over 256: 0 of 715


## 4. Pipeline 1 — Baseline dense retrieval (bi-encoder)
* Every chunk is embedded **once, offline** by the bi-encoder into a 384-d vector.
* Vectors are L2-normalised, so the inner product equals **cosine similarity**.
* `faiss.IndexFlatIP` performs an exact (brute-force) inner-product search — no approximation, so ranks are exact.
* At query time only the query is embedded; FAISS returns the **Top 10**. Pipeline 1 passes the first **3** to the LLM.

In [6]:
bi_encoder = SentenceTransformer(BI_ENCODER, device="cpu")
print("bi-encoder max_seq_length:", bi_encoder.max_seq_length)

t0 = time.perf_counter()
chunk_emb = bi_encoder.encode(chunk_texts, batch_size=64, normalize_embeddings=True,
                              convert_to_numpy=True, show_progress_bar=True).astype("float32")
index_time = time.perf_counter() - t0

index = faiss.IndexFlatIP(chunk_emb.shape[1])
index.add(chunk_emb)
print(f"Indexed {index.ntotal} chunks, dim={chunk_emb.shape[1]}, offline encoding took {index_time:.1f}s")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 50890.95it/s]

bi-encoder max_seq_length: 256


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Batches:   8%|▊         | 1/12 [00:01<00:12,  1.12s/it]

...

Batches:  75%|███████▌  | 9/12 [00:09<00:02,  1.03it/s]

Batches:  83%|████████▎ | 10/12 [00:09<00:01,  1.07it/s]

Batches:  92%|█████████▏| 11/12 [00:10<00:00,  1.24it/s]

Batches: 100%|██████████| 12/12 [00:10<00:00,  1.14it/s]

Indexed 715 chunks, dim=384, offline encoding took 10.6s


In [7]:
def retrieve(query: str, k: int = TOP_K_RETRIEVE):
    # Stage 1 (shared by both pipelines): embed query, cosine search, return top-k candidates.
    q = bi_encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    scores, ids = index.search(q, k)
    return [{"chunk_id": int(i), "bi_rank": r + 1, "bi_score": float(s)}
            for r, (i, s) in enumerate(zip(ids[0], scores[0]))]

def baseline_top3(candidates):
    # Pipeline 1: simply keep the first TOP_K_CONTEXT candidates in bi-encoder order.
    return candidates[:TOP_K_CONTEXT]

## 5. Pipeline 2 — Cross-encoder re-ranking
* Takes the **same 10 candidates** from Stage 1.
* Each `(query, chunk)` pair is fed **jointly** to the cross-encoder (`[CLS] query [SEP] chunk [SEP]`), which outputs one relevance logit.
* Candidates are re-sorted by that score (descending) and the new **Top 3** go to the LLM.

In [8]:
cross_encoder = CrossEncoder(CROSS_ENCODER, device="cpu", max_length=512)

def rerank(query: str, candidates):
    pairs = [(query, chunk_texts[c["chunk_id"]]) for c in candidates]
    ce_scores = cross_encoder.predict(pairs, batch_size=16, show_progress_bar=False)
    out = [dict(c, ce_score=float(s)) for c, s in zip(candidates, ce_scores)]
    out.sort(key=lambda c: c["ce_score"], reverse=True)
    for r, c in enumerate(out):
        c["ce_rank"] = r + 1
    return out

def reranked_top3(reranked):
    return reranked[:TOP_K_CONTEXT]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3368.79it/s]

## 6. LLM generation — identical prompt for both pipelines
The prompt is **strictly grounded**: the model must answer only from the supplied context and say so if the context is insufficient.
This matters for the experiment — Gemini already knows a lot about the AI Act, and without this instruction it could
"rescue" a bad retrieval from its own memory and hide the failure. Temperature is 0 for reproducibility.

In [9]:
llm = genai.Client(api_key=os.getenv("GEMINI_API_KEY"),
                   http_options=types.HttpOptions(timeout=90_000))   # ms; never hang on a stuck request

PROMPT_TEMPLATE = '''You are a legal assistant answering questions about the EU Artificial Intelligence Act (Regulation (EU) 2024/1689).
Answer the question using ONLY the context passages below. Do not use outside knowledge.
If the context does not contain the information needed, reply exactly: "The provided context does not contain the answer."
Be concise (max 5 sentences), and cite the passage label (e.g. [Article 5 — ...]) for every claim.

Context:
{context}

Question: {question}

Answer:'''

def build_prompt(question, top_chunks):
    context = "\n\n".join(f"Passage {i+1}: {chunk_texts[c['chunk_id']]}" for i, c in enumerate(top_chunks))
    return PROMPT_TEMPLATE.format(context=context, question=question)

# Answers are cached on disk (keyed by model + exact prompt), so re-running the notebook reproduces the same
# answers without spending API quota. Delete llm_cache.json to force fresh generations.
CACHE_PATH = "llm_cache.json"
try:
    llm_cache = json.load(open(CACHE_PATH, encoding="utf-8"))
except FileNotFoundError:
    llm_cache = {}

def generate(prompt, retries=4):
    # Returns (answer, latency_ms). The latency stored is that of the original API call, so cached re-runs
    # still report real generation times in the latency table.
    key = f"{LLM_MODEL}::{prompt}"
    if key in llm_cache:
        return llm_cache[key]["answer"], llm_cache[key]["latency_ms"]
    for attempt in range(retries):
        try:
            t0 = time.perf_counter()
            resp = llm.models.generate_content(
                model=LLM_MODEL, contents=prompt,
                config=types.GenerateContentConfig(temperature=0.0))
            llm_cache[key] = {"answer": resp.text.strip(), "latency_ms": (time.perf_counter() - t0) * 1000}
            json.dump(llm_cache, open(CACHE_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
            time.sleep(LLM_PAUSE_S)                       # stay under free-tier requests-per-minute limits
            return llm_cache[key]["answer"], llm_cache[key]["latency_ms"]
        except Exception as e:
            if "free_tier_requests" in str(e) and "PerDay" in str(e):
                raise RuntimeError("Daily free-tier quota exhausted for this model; set GEMINI_MODEL to another model.") from e
            if re.match(r"(400|401|403|404)", str(e)):        # bad request / key / model name: retrying cannot help
                raise
            wait = 2 ** attempt * 5                       # transient errors (503 overloaded, per-minute 429): back off
            print(f"  LLM error ({type(e).__name__}: {str(e)[:80]}), retrying in {wait}s")
            time.sleep(wait)
    return "[LLM call failed]", float("nan")

## 7. Side-by-side comparison helper
`compare(query, gold)` runs both pipelines on one query and returns everything needed for the analysis:
* a **rank table** of the 10 candidates (bi-encoder rank → cross-encoder rank),
* where the **gold chunk** (the one containing the decisive clause, identified by a distinctive phrase) sits in each ranking,
* **Answer 1** (baseline Top 3) and **Answer 2** (re-ranked Top 3), plus stage timings.

In [10]:
def is_gold(chunk_id, gold_phrases):
    t = chunk_texts[chunk_id].lower()
    return any(g.lower() in t for g in gold_phrases)

def compare(query, gold_phrases=(), with_llm=True, show=True):
    t0 = time.perf_counter(); cands = retrieve(query);       t1 = time.perf_counter()
    reranked = rerank(query, cands);                          t2 = time.perf_counter()
    top_a, top_b = baseline_top3(cands), reranked_top3(reranked)

    table = pd.DataFrame([{
        "bi_rank": c["bi_rank"], "ce_rank": c["ce_rank"], "shift": c["bi_rank"] - c["ce_rank"],
        "bi_score": round(c["bi_score"], 3), "ce_score": round(c["ce_score"], 2),
        "gold": "★" if is_gold(c["chunk_id"], gold_phrases) else "",
        "chunk_id": c["chunk_id"], "label": chunks[c["chunk_id"]]["label"],
        "snippet": chunk_texts[c["chunk_id"]][len(chunks[c["chunk_id"]]["label"]) + 3:][:90] + "…",
    } for c in reranked]).sort_values("bi_rank").reset_index(drop=True)

    gold_bi = [c["bi_rank"] for c in cands if is_gold(c["chunk_id"], gold_phrases)]
    gold_ce = [c["ce_rank"] for c in reranked if is_gold(c["chunk_id"], gold_phrases)]
    res = {"query": query, "table": table,
           "gold_bi_rank": min(gold_bi) if gold_bi else None,
           "gold_ce_rank": min(gold_ce) if gold_ce else None,
           "t_retrieve_ms": (t1 - t0) * 1000, "t_rerank_ms": (t2 - t1) * 1000}
    if with_llm:
        for i, top in ((1, top_a), (2, top_b)):
            res[f"answer_{i}"], res[f"t_llm_{i}_ms"] = generate(build_prompt(query, top))
    if show:
        print(f"Q: {query}")
        print(f"Gold chunk rank — baseline: {res['gold_bi_rank']}  →  re-ranked: {res['gold_ce_rank']}"
              f"   | retrieve {res['t_retrieve_ms']:.0f} ms, re-rank {res['t_rerank_ms']:.0f} ms")
        display(table)
        if with_llm:
            print("ANSWER 1 (baseline top-3):\n" + res["answer_1"] + "\n")
            print("ANSWER 2 (re-ranked top-3):\n" + res["answer_2"])
    return res

# Smoke test of the retrieval stages (LLM step skipped here to save free-tier quota for the stress tests).
_ = compare("Which AI practices are prohibited under the AI Act?", gold_phrases=["Prohibited AI practices"], with_llm=False)

Q: Which AI practices are prohibited under the AI Act?
Gold chunk rank — baseline: 1  →  re-ranked: 1   | retrieve 26 ms, re-rank 188 ms


,bi_rank,ce_rank,shift,bi_score,ce_score,gold,chunk_id,label,snippet
0,1,1,0,0.762,8.38,★,316,Article 5 — Prohibited AI practices,"1. The following AI practices shall be prohibited: (a) the placing on the market, the putt…"
1,2,7,-5,0.706,-0.45,,6,Recital 5,"At the same time, depending on the circumstances regarding its specific application, use, …"
2,3,2,1,0.705,6.39,,48,Recital 28,"Aside from the many beneficial uses of AI, it can also be misused and provide novel and po…"
3,4,6,-2,0.703,-0.43,,8,Recital 7,In order to ensure a consistent and high level of protection of public interests as regard…
4,5,5,0,0.695,1.17,,294,Article 2 — Scope,Such activities shall be conducted in accordance with applicable Union law. Testing in rea…
5,6,3,3,0.694,5.33,,76,Recital 44,That prohibition should not cover AI systems placed on the market strictly for medical or …
6,7,9,-2,0.684,-6.35,,624,Article 102 — Amendment to Regulation (EC) No 300/2008,(*) Regulation (EU) 2024/1689 of the European Parliament and of the Council of 13 June 202…
7,8,4,4,0.684,1.58,,13,Recital 9,"Furthermore, the obligations placed on various operators involved in the AI value chain un…"
8,9,10,-1,0.679,-6.60,,630,Article 105 — Amendment to Directive 2014/90/EU,(*) Regulation (EU) 2024/1689 of the European Parliament and of the Council of 13 June 202…
9,10,8,2,0.679,-6.31,,634,Article 107 — Amendment to Regulation (EU) 2018/858,(*) Regulation (EU) 2024/1689 of the European Parliament and of the Council of 13 June 202…


## 8. Failure-mode stress tests
73 queries were written and labelled during development (Section 8b evaluates all of them). The cases below are the clearest
example(s) of each failure category in the assignment. For every query the **gold chunk** is the one containing the decisive
clause; it is identified by a short phrase that only appears in that clause, so its rank can be tracked objectively in both pipelines.

| Assignment category | Cases |
|---|---|
| Keyword trap / distractor | F1 (date-heavy boilerplate distractors), F2 (lexical distractors: "label", "images") |
| Exceptions & corner cases | F3, F4 (baseline gets the general rule, misses the "shall not apply …" exception) |
| Negation & constraints / role limits | F5 (negation), F6 (role constraint) |
| Other: vital chunk buried at rank 4–10 | F7 (and every case above: all are promoted from rank 4–10 into the Top 3) |

Two **control / limitation** cases are included too, so the picture stays honest: one where re-ranking *hurts*, and one where
the gold chunk never reaches the Top 10 (re-ranking cannot recover what Stage 1 did not retrieve).

In [11]:
FAILURE_QUERIES = [
    {"id": "F1", "category": "Distractor (boilerplate) + exception",
     "query": "From what date do the bans on prohibited AI practices apply?",
     "gold": ["2 February 2025"],
     "why": "General rule: the Act applies from 2 Aug 2026; exception: prohibitions apply earlier (2 Feb 2025, Recital 179 / Art. 113). "
            "Distractors: nine identical citations of the Act's full title with dates (Articles 102–110)."},
    {"id": "F2", "category": "Keyword trap (lexical overlap, different terminology)",
     "query": "Do companies have to label AI-generated images?",
     "gold": ["detectable as artificially generated"],
     "why": "Article 50(2) never says 'label': outputs must be 'marked in a machine-readable format and detectable as artificially "
            "generated'. Distractors share the query's words ('images', a provider's 'name or trademark')."},
    {"id": "F3", "category": "Exception (general rule retrieved, exception missed)",
     "query": "Do deployers of emotion recognition systems always have to inform the people exposed to them?",
     "gold": ["permitted by law to detect, prevent or investigate criminal offences"],
     "why": "Article 50(3): deployers must inform people … 'This obligation shall not apply to AI systems used for biometric "
            "categorisation and emotion recognition, which are permitted by law to detect, prevent or investigate criminal offences'."},
    {"id": "F4", "category": "Exception / corner case (urgency procedure)",
     "query": "Can police start using real-time facial recognition before receiving authorisation?",
     "gold": ["within 24 hours"],
     "why": "General rule: each use needs prior authorisation; exception: in duly justified urgency, use may start without it if "
            "authorisation is requested within 24 hours (Art. 5(3), Recital 35)."},
    {"id": "F5", "category": "Negation (\"not covered\")",
     "query": "Which biometric categorisation activities are not covered by the prohibition?",
     "gold": ["does not cover any labelling or filtering"],
     "why": "Article 5(1)(g) carve-out: labelling/filtering of lawfully acquired biometric datasets is not prohibited."},
    {"id": "F6", "category": "Role constraint",
     "query": "When does a deployer become the provider of a high-risk AI system?",
     "gold": ["modify the intended purpose"],
     "why": "Article 25(1): a deployer is treated as a provider if it rebrands, substantially modifies, or changes the intended purpose."},
    {"id": "F7", "category": "Buried chunk (rank 4–10)",
     "query": "When is a general-purpose AI model presumed to have systemic risk?",
     "gold": ["presumed to have high impact capabilities"],
     "why": "The presumption (training compute > 10^25 FLOPs) is only in Article 51(2)."},
    {"id": "C1", "category": "Control: re-ranking hurts",
     "query": "Can AI be used to manipulate people's behaviour without them noticing?",
     "gold": ["subliminal techniques"],
     "why": "Article 5(1)(a) bans subliminal/manipulative techniques; Recital 29 explains them in very similar words."},
    {"id": "C2", "category": "Limitation: gold not in Top 10",
     "query": "Can an AI system listed in Annex III be exempted from being high-risk?",
     "gold": ["shall not be considered to be high-risk where it does not pose a significant risk"],
     "why": "Article 6(3) derogation — the bi-encoder never retrieves it, so the cross-encoder never sees it."},
]

results = {}
for fq in FAILURE_QUERIES:
    print("=" * 110)
    print(f"{fq['id']} — {fq['category']}\nDecisive clause: {fq['why']}")
    results[fq["id"]] = compare(fq["query"], fq["gold"])

F1 — Distractor (boilerplate) + exception
Decisive clause: General rule: the Act applies from 2 Aug 2026; exception: prohibitions apply earlier (2 Feb 2025, Recital 179 / Art. 113). Distractors: nine identical citations of the Act's full title with dates (Articles 102–110).
Q: From what date do the bans on prohibited AI practices apply?
Gold chunk rank — baseline: 7  →  re-ranked: 1   | retrieve 8 ms, re-rank 168 ms


,bi_rank,ce_rank,shift,bi_score,ce_score,gold,chunk_id,label,snippet
0,1,6,-5,0.688,-7.54,,624,Article 102 — Amendment to Regulation (EC) No 300/2008,(*) Regulation (EU) 2024/1689 of the European Parliament and of the Council of 13 June 202…
1,2,4,-2,0.686,-7.07,,630,Article 105 — Amendment to Directive 2014/90/EU,(*) Regulation (EU) 2024/1689 of the European Parliament and of the Council of 13 June 202…
2,3,10,-7,0.685,-7.62,,628,Article 104 — Amendment to Regulation (EU) No 168/2013,(*) Regulation (EU) 2024/1689 of the European Parliament and of the Council of 13 June 202…
3,4,8,-4,0.683,-7.61,,626,Article 103 — Amendment to Regulation (EU) No 167/2013,(*) Regulation (EU) 2024/1689 of the European Parliament and of the Council of 13 June 202…
4,5,5,0,0.683,-7.52,,634,Article 107 — Amendment to Regulation (EU) 2018/858,(*) Regulation (EU) 2024/1689 of the European Parliament and of the Council of 13 June 202…
5,6,9,-3,0.681,-7.62,,641,Article 109 — Amendment to Regulation (EU) 2019/2144,(*) Regulation (EU) 2024/1689 of the European Parliament and of the Council of 13 June 202…
6,7,1,6,0.680,7.30,★,286,Recital 179,"This Regulation should apply from 2 August 2026. However, taking into account the unaccept…"
7,8,7,1,0.679,-7.56,,632,Article 106 — Amendment to Directive (EU) 2016/797,(*) Regulation (EU) 2024/1689 of the European Parliament and of the Council of 13 June 202…
8,9,2,7,0.672,2.23,,287,Recital 179,The AI Office should ensure that classification rules and procedures are up to date in lig…
9,10,3,7,0.665,1.50,,316,Article 5 — Prohibited AI practices,"1. The following AI practices shall be prohibited: (a) the placing on the market, the putt…"


ANSWER 1 (baseline top-3):
The provided context does not contain the answer.

ANSWER 2 (re-ranked top-3):
The prohibitions on prohibited AI practices apply from 2 February 2025 [Recital 179].
F2 — Keyword trap (lexical overlap, different terminology)
Decisive clause: Article 50(2) never says 'label': outputs must be 'marked in a machine-readable format and detectable as artificially generated'. Distractors share the query's words ('images', a provider's 'name or trademark').


Q: Do companies have to label AI-generated images?
Gold chunk rank — baseline: 10  →  re-ranked: 3   | retrieve 9 ms, re-rank 163 ms


,bi_rank,ce_rank,shift,bi_score,ce_score,gold,chunk_id,label,snippet
0,1,6,-5,0.540,-7.53,,390,Article 25 — Responsibilities along the AI value chain,(b) the high-risk AI system is put into service under the name or trademark of the product…
1,2,4,-2,0.525,-3.29,,74,Recital 43,"The placing on the market, the putting into service for that specific purpose, or the use …"
2,3,5,-2,0.524,-4.18,,319,Article 5 — Prohibited AI practices,"(e) the placing on the market, the putting into service for this specific purpose, or the …"
3,4,9,-5,0.520,-8.52,,149,Recital 84,"To ensure legal certainty, it is necessary to clarify that, under certain specific conditi…"
4,5,1,4,0.514,2.64,,219,Recital 134,"Further to the technical solutions employed by the providers of the AI system, deployers w…"
5,6,8,-2,0.506,-8.30,,295,Article 3 — Definitions,"For the purposes of this Regulation, the following definitions apply: (1) ‘AI system’ mean…"
6,7,7,0,0.501,-8.13,,389,Article 25 — Responsibilities along the AI value chain,3. In the case of high-risk AI systems that are safety components of products covered by t…
7,8,10,-2,0.493,-8.83,,387,Article 25 — Responsibilities along the AI value chain,"1. Any distributor, importer, deployer or other third-party shall be considered to be a pr…"
8,9,2,7,0.492,-1.11,,461,Article 50 — Transparency obligations for providers and deployers of certain AI systems,This obligation shall not apply to AI systems used for biometric categorisation and emotio…
9,10,3,7,0.490,-2.12,★,459,Article 50 — Transparency obligations for providers and deployers of certain AI systems,1. Providers shall ensure that AI systems intended to interact directly with natural perso…


...

,bi_rank,ce_rank,shift,bi_score,ce_score,gold,chunk_id,label,snippet
0,1,1,0,0.634,6.35,,49,Recital 29,AI-enabled manipulative techniques can be used to persuade persons to engage in unwanted b…
1,2,4,-2,0.609,1.71,★,316,Article 5 — Prohibited AI practices,"1. The following AI practices shall be prohibited: (a) the placing on the market, the putt…"
2,3,3,0,0.595,2.09,,100,Recital 57,AI systems used to monitor the performance and behaviour of such persons may also undermin…
3,4,2,2,0.588,3.69,,50,Recital 29,"This could be facilitated, for example, by machine-brain interfaces or virtual reality as …"
4,5,9,-4,0.514,-3.09,,459,Article 50 — Transparency obligations for providers and deployers of certain AI systems,1. Providers shall ensure that AI systems intended to interact directly with natural perso…
5,6,8,-2,0.512,-2.80,,6,Recital 5,"At the same time, depending on the circumstances regarding its specific application, use, …"
6,7,7,0,0.503,-0.51,,75,Recital 44,There are serious concerns about the scientific basis of AI systems aiming to identify or …
7,8,6,2,0.495,-0.30,,215,Recital 132,Certain AI systems intended to interact with natural persons or to generate content may po…
8,9,5,4,0.494,0.28,,48,Recital 28,"Aside from the many beneficial uses of AI, it can also be misused and provide novel and po…"
9,10,10,0,0.487,-4.86,,17,Recital 12,The notion of ‘AI system’ in this Regulation should be clearly defined and should be close…


ANSWER 1 (baseline top-3):
Yes, AI can be used to manipulate people's behaviour without them consciously noticing [Recital 29]. Specifically, AI systems can deploy subliminal components—such as audio, image, or video stimuli—that are beyond human perception [Recital 29]. They can also employ manipulative or deceptive techniques to subvert or impair a person's autonomy, decision-making, or free choices in ways that people are not consciously aware of [Recital 29]. Furthermore, AI systems can deploy subliminal techniques beyond a person’s consciousness with the objective or effect of materially distorting their behaviour, causing them to make decisions they would not otherwise have taken [Article 5 — Prohibited AI practices].

ANSWER 2 (re-ranked top-3):
Yes, AI systems can manipulate people's behaviour without them noticing [Recital 29]. Specifically, such systems can deploy subliminal components—such as audio, image, or video stimuli—that are beyond human perception [Recital 29]. They 

Q: Can an AI system listed in Annex III be exempted from being high-risk?
Gold chunk rank — baseline: None  →  re-ranked: None   | retrieve 12 ms, re-rank 228 ms


,bi_rank,ce_rank,shift,bi_score,ce_score,gold,chunk_id,label,snippet
0,1,2,-1,0.768,7.01,,334,Article 7 — Amendments to Annex III,1. The Commission is empowered to adopt delegated acts in accordance with Article 97 to am…
1,2,5,-3,0.745,6.64,,440,Article 43 — Conformity assessment,"2. For high-risk AI systems referred to in points 2 to 8 of Annex III, providers shall fol…"
2,3,7,-4,0.736,5.88,,329,Article 6 — Classification rules for high-risk AI systems,1. Irrespective of whether an AI system is placed on the market or put into service indepe…
3,4,1,3,0.730,7.07,,361,Article 14 — Human oversight,"5. For high-risk AI systems referred to in point 1(a) of Annex III, the measures referred …"
4,5,10,-5,0.728,1.22,,333,Article 6 — Classification rules for high-risk AI systems,"8. Any amendment to the conditions laid down in paragraph 3, second subparagraph, adopted …"
5,6,3,3,0.728,6.74,,579,Article 80 — Procedure for dealing with AI systems classified by the provider as non-high-risk in application of Ann...,5. The provider shall ensure that all appropriate corrective action is taken in respect of…
6,7,4,3,0.727,6.70,,458,Article 49 — Registration,Only the Commission and national authorities referred to in Article 74(8) shall have acces…
7,8,8,0,0.723,3.22,,271,Recital 166,It is important that AI systems related to products that are not high-risk in accordance w…
8,9,6,3,0.715,6.37,,337,Article 7 — Amendments to Annex III,(j) the magnitude and likelihood of benefit of the deployment of the AI system for individ…
9,10,9,1,0.711,2.72,,87,Recital 52,"As regards stand-alone AI systems, namely high-risk AI systems other than those that are s…"


ANSWER 1 (baseline top-3):
The provided context does not contain the answer.

ANSWER 2 (re-ranked top-3):
The provided context does not contain the answer.


In [12]:
# Summary of rank shifts for the gold chunk (None = not in the Top 10).
summary = pd.DataFrame([{
    "id": fq["id"], "category": fq["category"],
    "gold rank: baseline": results[fq["id"]]["gold_bi_rank"],
    "gold rank: re-ranked": results[fq["id"]]["gold_ce_rank"],
    "in baseline Top 3": bool(results[fq["id"]]["gold_bi_rank"] and results[fq["id"]]["gold_bi_rank"] <= 3),
    "in re-ranked Top 3": bool(results[fq["id"]]["gold_ce_rank"] and results[fq["id"]]["gold_ce_rank"] <= 3),
} for fq in FAILURE_QUERIES])
summary

,id,category,gold rank: baseline,gold rank: re-ranked,in baseline Top 3,in re-ranked Top 3
0,F1,Distractor (boilerplate) + exception,7.0,1.0,False,True
1,F2,"Keyword trap (lexical overlap, different terminology)",10.0,3.0,False,True
2,F3,"Exception (general rule retrieved, exception missed)",9.0,2.0,False,True
3,F4,Exception / corner case (urgency procedure),7.0,2.0,False,True
4,F5,"Negation (""not covered"")",9.0,2.0,False,True
5,F6,Role constraint,7.0,1.0,False,True
6,F7,Buried chunk (rank 4–10),7.0,1.0,False,True
7,C1,Control: re-ranking hurts,2.0,4.0,True,False
8,C2,Limitation: gold not in Top 10,NaN,NaN,False,False


### 8b. Aggregate retrieval evaluation (all labelled queries)
Hand-picked failure cases can be cherry-picked, so both rankings are also scored over **every query labelled during development**
(`data/eval_queries.json`). Only queries whose gold phrase occurs in at most 3 chunks are kept, so "gold" stays unambiguous.
Metrics: **Hit@k** = gold chunk within the Top k (Top 3 is what the LLM sees); **MRR@10** = mean of 1/rank (0 if not in Top 10).

*Caveat:* these queries were written by us while probing for failures, so they over-represent hard cases; the numbers show the
*direction and size* of the effect on this document, not a general benchmark result.

In [13]:
eval_set = json.load(open("data/eval_queries.json", encoding="utf-8"))
rows = []
for e in eval_set:
    gold_ids = {i for i in range(len(chunks)) if is_gold(i, e["gold"])}
    if not gold_ids or len(gold_ids) > 3:
        continue
    cands = retrieve(e["query"]); rr = rerank(e["query"], cands)
    rows.append({"id": e["id"], "query": e["query"],
                 "baseline rank": min([c["bi_rank"] for c in cands if c["chunk_id"] in gold_ids], default=None),
                 "re-ranked rank": min([c["ce_rank"] for c in rr if c["chunk_id"] in gold_ids], default=None)})
ev = pd.DataFrame(rows)
n = len(ev)

def hit(col, k): return int((ev[col] <= k).sum())
def mrr(col): return float((1 / ev[col]).fillna(0).mean())

metrics = pd.DataFrame({
    "Baseline (bi-encoder)": [f"{hit('baseline rank', 1)}/{n}", f"{hit('baseline rank', 3)}/{n}", f"{mrr('baseline rank'):.3f}"],
    "Re-ranked (cross-encoder)": [f"{hit('re-ranked rank', 1)}/{n}", f"{hit('re-ranked rank', 3)}/{n}", f"{mrr('re-ranked rank'):.3f}"],
}, index=["Hit@1", "Hit@3 (what the LLM sees)", "MRR@10"])
display(metrics)

found = ev.dropna()
print(f"Gold chunk retrieved in Top 10 at all (ceiling for re-ranking): {len(found)}/{n}")
print(f"Among those: re-ranking improved the gold rank in {(found['re-ranked rank'] < found['baseline rank']).sum()}, "
      f"kept it in {(found['re-ranked rank'] == found['baseline rank']).sum()}, worsened it in "
      f"{(found['re-ranked rank'] > found['baseline rank']).sum()}")
print(f"Promoted into the Top 3: {((found['baseline rank'] > 3) & (found['re-ranked rank'] <= 3)).sum()} | "
      f"pushed out of the Top 3: {((found['baseline rank'] <= 3) & (found['re-ranked rank'] > 3)).sum()}")

,Baseline (bi-encoder),Re-ranked (cross-encoder)
Hit@1,14/65,23/65
Hit@3 (what the LLM sees),34/65,41/65
MRR@10,0.366,0.496


Gold chunk retrieved in Top 10 at all (ceiling for re-ranking): 45/65
Among those: re-ranking improved the gold rank in 24, kept it in 16, worsened it in 5
Promoted into the Top 3: 10 | pushed out of the Top 3: 3


## 9. Latency profiling
Each stage is timed separately with `time.perf_counter()` after a warm-up run, repeating every query 20 times (CPU only, Intel Core Ultra 5 225H):

* **Baseline retrieval** = embed the query with the bi-encoder + FAISS search over all chunks → Top 10
* **Re-ranking** = 10 cross-encoder forward passes (one per `(query, chunk)` pair) → Top 3
* **Pipeline 2 retrieval** = baseline retrieval + re-ranking
* **LLM generation** is reported separately (network-bound and highly variable, identical for both pipelines).

In [14]:
REPEATS = 20
profile_queries = [fq["query"] for fq in FAILURE_QUERIES]

for q in profile_queries[:2]:                     # warm-up (model weights into cache, thread pools started)
    rerank(q, retrieve(q))

rows = []
for q in profile_queries:
    for _ in range(REPEATS):
        t0 = time.perf_counter()
        qv = bi_encoder.encode([q], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
        t1 = time.perf_counter()
        index.search(qv, TOP_K_RETRIEVE)
        t2 = time.perf_counter()
        cands = retrieve(q)
        t3 = time.perf_counter()
        rerank(q, cands)
        t4 = time.perf_counter()
        rows.append({"query encode": (t1 - t0) * 1e3, "FAISS search": (t2 - t1) * 1e3,
                     "Baseline retrieval (P1)": (t3 - t2) * 1e3, "Cross-encoder re-rank (10 pairs)": (t4 - t3) * 1e3})

lat = pd.DataFrame(rows)
lat["Re-rank pipeline retrieval (P2)"] = lat["Baseline retrieval (P1)"] + lat["Cross-encoder re-rank (10 pairs)"]
latency_table = lat.describe(percentiles=[0.5, 0.95]).T[["mean", "50%", "95%", "min", "max"]].round(2)
latency_table.columns = ["mean (ms)", "median (ms)", "p95 (ms)", "min (ms)", "max (ms)"]
display(latency_table)

llm_times = [results[k][f"t_llm_{i}_ms"] for k in results for i in (1, 2)]
print(f"LLM generation (Gemini, network): mean {np.mean(llm_times):.0f} ms, median {np.median(llm_times):.0f} ms "
      f"over {len(llm_times)} calls")

# End-to-end view: retrieval medians + the median LLM call (generation time is the same for both pipelines).
llm_med = float(np.median(llm_times))
p1 = latency_table.loc["Baseline retrieval (P1)", "median (ms)"]
p2 = latency_table.loc["Re-rank pipeline retrieval (P2)", "median (ms)"]
display(pd.DataFrame({"retrieval (ms)": [p1, p2], "LLM generation (ms)": [llm_med, llm_med],
                      "end-to-end (ms)": [p1 + llm_med, p2 + llm_med]},
                     index=["Pipeline 1 — baseline", "Pipeline 2 — re-ranking"]).round(0))
print(f"End-to-end overhead of re-ranking: +{(p2 - p1) / (p1 + llm_med) * 100:.1f}%")
print(f"Offline indexing (one-time): {index_time:.1f} s for {len(chunks)} chunks")
ratio = latency_table.loc["Re-rank pipeline retrieval (P2)", "median (ms)"] / latency_table.loc["Baseline retrieval (P1)", "median (ms)"]
print(f"Re-ranking makes retrieval ~{ratio:.1f}x slower, but adds only "
      f"{latency_table.loc['Cross-encoder re-rank (10 pairs)', 'median (ms)']:.0f} ms — small next to the LLM call.")

,mean (ms),median (ms),p95 (ms),min (ms),max (ms)
query encode,8.45,7.54,10.31,5.88,101.44
FAISS search,0.11,0.11,0.13,0.09,0.28
Baseline retrieval (P1),7.74,6.84,10.54,5.40,69.45
Cross-encoder re-rank (10 pairs),168.13,155.02,226.53,124.25,752.37
Re-rank pipeline retrieval (P2),175.87,162.23,233.39,130.44,762.13


LLM generation (Gemini, network): mean 39434 ms, median 34924 ms over 18 calls


,retrieval (ms),LLM generation (ms),end-to-end (ms)
Pipeline 1 — baseline,7.0,34924.0,34930.0
Pipeline 2 — re-ranking,162.0,34924.0,35086.0


End-to-end overhead of re-ranking: +0.4%
Offline indexing (one-time): 10.6 s for 715 chunks
Re-ranking makes retrieval ~23.7x slower, but adds only 155 ms — small next to the LLM call.


## 10. Scaling experiment — why not cross-encode the whole corpus?
A cross-encoder needs the query to compute anything, so **nothing can be pre-computed**: ranking the full corpus means one transformer
forward pass for *every* chunk at query time. Below we actually do this for our corpus and extrapolate linearly to larger collections.
We also check whether full cross-encoder ranking would have rescued the limitation case **C2**.

In [15]:
q = FAILURE_QUERIES[-1]["query"]           # C2 — the case where the gold chunk was outside the bi-encoder Top 10

t0 = time.perf_counter()
full_scores = cross_encoder.predict([(q, t) for t in chunk_texts], batch_size=64, show_progress_bar=False)
t_full_ce = time.perf_counter() - t0

t0 = time.perf_counter(); retrieve(q); t_bi = time.perf_counter() - t0

order = np.argsort(-full_scores)
gold_ids = [i for i in range(len(chunks)) if is_gold(i, FAILURE_QUERIES[-1]["gold"])]
full_rank = int(np.where(np.isin(order, gold_ids))[0].min()) + 1
bi_full = (chunk_emb @ bi_encoder.encode([q], normalize_embeddings=True)[0])
bi_rank_full = int(np.where(np.isin(np.argsort(-bi_full), gold_ids))[0].min()) + 1
print(f"C2 gold chunk rank — bi-encoder over full corpus: #{bi_rank_full} | cross-encoder over full corpus: #{full_rank}")

def fmt_secs(s):
    return f"{s * 1e3:.0f} ms" if s < 1 else f"{s:.1f} s" if s < 120 else f"{s / 60:.0f} min" if s < 7200 else f"{s / 3600:.1f} h"

per_pair = t_full_ce / len(chunks)
scale = pd.DataFrame([{
    "corpus size (chunks)": f"{n:,}" + (" (this document, measured)" if n == len(chunks) else " (extrapolated)"),
    "Pipeline 2: bi-encoder Top 10 + re-rank 10": fmt_secs(t_bi + 10 * per_pair) if n == len(chunks) else "≈ same (ANN search is sub-linear)",
    "cross-encoder over every chunk": fmt_secs(per_pair * n),
} for n in [len(chunks), 10_000, 100_000, 1_000_000, 10_000_000]])
print(f"Measured: {len(chunks)} cross-encoder pairs in {t_full_ce:.2f} s ({per_pair * 1e3:.2f} ms/pair, CPU, batch 64) "
      f"vs. bi-encoder retrieval {t_bi * 1e3:.0f} ms")
display(scale)
print("Bi-encoder chunk vectors are computed once offline; at query time only the query is encoded, and an approximate-"
      "nearest-neighbour index (e.g. FAISS HNSW/IVF) keeps search in the millisecond range even for millions of vectors. "
      "The cross-encoder cost grows linearly with the corpus, for every single query.")

C2 gold chunk rank — bi-encoder over full corpus: #50 | cross-encoder over full corpus: #1
Measured: 715 cross-encoder pairs in 11.42 s (15.98 ms/pair, CPU, batch 64) vs. bi-encoder retrieval 7 ms


,corpus size (chunks),Pipeline 2: bi-encoder Top 10 + re-rank 10,cross-encoder over every chunk
0,"715 (this document, measured)",167 ms,11.4 s
1,"10,000 (extrapolated)",≈ same (ANN search is sub-linear),3 min
2,"100,000 (extrapolated)",≈ same (ANN search is sub-linear),27 min
3,"1,000,000 (extrapolated)",≈ same (ANN search is sub-linear),4.4 h
4,"10,000,000 (extrapolated)",≈ same (ANN search is sub-linear),44.4 h


Bi-encoder chunk vectors are computed once offline; at query time only the query is encoded, and an approximate-nearest-neighbour index (e.g. FAISS HNSW/IVF) keeps search in the millisecond range even for millions of vectors. The cross-encoder cost grows linearly with the corpus, for every single query.
